In [1]:
# This notebook requires you to have access to Kaggle T4x2 GPU

!pip install open3d
!pip install diffusers transformers accelerate trimesh pygltflib
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd Hunyuan3D-2
!pip install -r requirements.txt
%cd hy3dgen/texgen/custom_rasterizer
!python setup.py bdist_wheel
!pip install dist/custom_rasterizer*.whl
%cd ..
%cd ..
%cd ..

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 98.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 16.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.5 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have

In [2]:
!pip install fastapi uvicorn pyngrok nest-asyncio trimesh

In [3]:
import gc
import torch
import time


def clean_memory():
    # Clean CPU memory
    gc.collect()

    # Clean GPU memory (if available)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("GPU memory cleared.")
    else:
        print("No GPU available.")

In [4]:
from PIL import Image
import requests
import numpy as np

In [5]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN)

from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [7]:
## Model Loading

from hy3dgen.texgen import Hunyuan3DPaintPipeline
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline


clean_memory()

DEVICE_2 = torch.device(
    "cuda:1"
    if torch.cuda.device_count() > 1
    else ("cuda:0" if torch.cuda.is_available() else "cpu")
)

mesh_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-2",
    subfolder="hunyuan3d-dit-v2-0",
    variant="fp16",
    device=DEVICE_2,
)

paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained(
    "tencent/Hunyuan3D-2", subfolder="hunyuan3d-paint-v2-0-turbo"
)

print("Models Loaded")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
2026-06-10 08:19:17,891 - hy3dgen.shapgen - INFO - Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
INFO:hy3dgen.shapgen:Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2026-06-10 08:19:17,894 - hy3dgen.shapgen - INFO - Model path not exists, try to download from huggingface
INFO:hy3dgen.shapgen:Model path not exists, try to download from huggingface


GPU memory cleared.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

2026-06-10 08:19:56,419 - hy3dgen.shapgen - INFO - Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-dit-v2-0/model.fp16.safetensors
INFO:hy3dgen.shapgen:Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-dit-v2-0/model.fp16.safetensors


PointCrossAttentionEncoder INFO: pc_sharpedge_size is given, using pc_size=5120, pc_sharpedge_size=5120


...

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-paint-v2-0-turbo/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-paint-v2-0-turbo/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-paint-v2-0-turbo/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Expected types for unet: (<class 'diffusers_module

Models Loaded


In [8]:
import trimesh
import open3d as o3d


def preprocess_mesh(mesh):
    o3d_mesh = o3d.geometry.TriangleMesh()
    o3d_mesh.vertices = o3d.utility.Vector3dVector(mesh.vertices)
    o3d_mesh.triangles = o3d.utility.Vector3iVector(mesh.faces)

    target_faces = int(len(mesh.faces) * 0.1)
    simplified = o3d_mesh.simplify_quadric_decimation(target_faces)

    decimated_mesh_90 = trimesh.Trimesh(
        vertices=np.asarray(simplified.vertices),
        faces=np.asarray(simplified.triangles),
        process=False,
    )
    return decimated_mesh_90

In [9]:
from pathlib import Path


def run_model(image, name, output_folder):
    name = name.replace(" ", "_")
    img_mesh = mesh_pipeline(image=image)[0]
    clean_memory()
    decimated_mesh_90 = preprocess_mesh(img_mesh)
    paint_mesh = paint_pipeline(decimated_mesh_90, image=image)
    clean_memory()

    paint_mesh.export(f"{output_folder}/paint_mesh_{name}.glb")
    time.sleep(5)

In [10]:
IMAGE_PATH_JSON = "/kaggle/input/datasets/shivanshpachnanda/noaa-scraped-illus-links/exp-noaa-testing-image_sites.json"

import json

with open(IMAGE_PATH_JSON, "r") as file:
    json_links = json.load(file)

In [11]:
json_links = {
    key.split("/")[-1].replace("-", " ").lower(): value
    for key, value in json_links.items()
}
json_keys_list = list(json_links.keys())

In [12]:
# Handle images Code
from urllib.parse import urljoin

from rembg import remove

BASE_SITE = "https://www.fisheries.noaa.gov/"


def generate_model(name, output_folder):
    image_path = urljoin(BASE_SITE, json_links[name])
    img_data = requests.get(image_path).content

    with open(f"{str(output_folder)}/reference.jpg", "wb") as handler:
        handler.write(img_data)

    image = Image.open(f"{str(output_folder)}/reference.jpg").convert("RGBA")
    removed_image = remove(image)

    run_model(removed_image, name, output_folder)

In [13]:
# Ngrok Sending

from pygltflib import GLTF2
import os
import shutil


def extract_textures_and_copy_model(glb_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    gltf = GLTF2().load(glb_path)

    glb_filename = os.path.basename(glb_path)
    copied_glb_path = os.path.join(output_dir, glb_filename)
    shutil.copy2(glb_path, copied_glb_path)

    with open(glb_path, "rb") as f:
        content = f.read()

    def get_bin_chunk():
        magic = int.from_bytes(content[0:4], "little")
        assert magic == 0x46546C67  # b'glTF'
        json_len = int.from_bytes(content[12:16], "little")
        json_type = content[16:20]
        assert json_type == b"JSON"

        bin_offset = 20 + json_len
        bin_len = int.from_bytes(content[bin_offset : bin_offset + 4], "little")
        bin_type = content[bin_offset + 4 : bin_offset + 8]
        assert bin_type == b"BIN\x00"

        return content[bin_offset + 8 : bin_offset + 8 + bin_len]

    bin_chunk = get_bin_chunk()
    base_name = os.path.splitext(glb_filename)[0]

    for i, image in enumerate(gltf.images):
        if image.bufferView is None:
            print(f"Skipping image {i} (no bufferView)")
            continue

        buffer_view = gltf.bufferViews[image.bufferView]
        offset = buffer_view.byteOffset or 0
        length = buffer_view.byteLength

        image_data = bin_chunk[offset : offset + length]
        ext = "png" if image.mimeType == "image/png" else "jpg"
        out_path = os.path.join(output_dir, f"{base_name}_texture_{i}.{ext}")

        with open(out_path, "wb") as f:
            f.write(image_data)

In [14]:
import zipfile


def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                abs_path = os.path.join(root, file)
                rel_path = os.path.relpath(
                    abs_path, folder_path
                )  # relative path inside zip
                zipf.write(abs_path, arcname=rel_path)

In [17]:
import json
from datetime import datetime

GIST_ID = "3c709360165ff302c242a38407cb03ad"


def update_endpoint(endpoint_url):
    content = json.dumps(
        {"endpoint": endpoint_url, "last_checked": datetime.utcnow().isoformat() + "Z"},
        indent=4,
    )

    response = requests.patch(
        f"https://api.github.com/gists/{GIST_ID}",
        headers={
            "Authorization": f"Bearer {GITHUB_TOKEN}",
            "Accept": "application/vnd.github+json",
        },
        json={"files": {"Endpoint.json": {"content": content}}},
    )

    response.raise_for_status()
    print("Gist updated successfully")

In [ ]:
from fastapi import FastAPI, Form
from fastapi.responses import FileResponse, JSONResponse
import nest_asyncio
import uvicorn
from fastapi.responses import PlainTextResponse


nest_asyncio.apply()

app = FastAPI()
PORT = 8000

OUTPUT_DIR = (
    "/kaggle/input/datasets/shivanshpachnanda/3d-model-short-dataset/OUTPUT_FOLDER"
)
MODEL_DIR = Path("/kaggle/working/models")
CACHE_OUTPUT_DIR = os.path.join("/kaggle/working/", "cache")
ZIP_PATH = "/kaggle/working/zip/output.zip"
Path(ZIP_PATH).parent.mkdir(exist_ok=True, parents=True)
Path(MODEL_DIR).mkdir(exist_ok=True, parents=True)


@app.get("/")
def read_root():
    return PlainTextResponse(
        "Welcome! POST a filename to /process-3d-file to get a ZIP with model + textures."
    )


@app.post("/process-3d-file")
def process_3d_file(concept: str = Form(...)):
    concept = concept.strip().lower().replace("_", " ")
    if os.path.exists(CACHE_OUTPUT_DIR):
        shutil.rmtree(CACHE_OUTPUT_DIR)
    os.makedirs(CACHE_OUTPUT_DIR, exist_ok=True)

    list_paths = os.listdir(OUTPUT_DIR)
    if (MODEL_DIR / f"paint_mesh_{concept.replace(' ', '_')}.glb").exists():
        glb_path = MODEL_DIR / f"paint_mesh_{concept.replace(' ', '_')}.glb"
    elif concept in json_keys_list:
        generate_model(concept, MODEL_DIR)
        glb_path = MODEL_DIR / f"paint_mesh_{concept.replace(' ', '_')}.glb"
    else:
        if concept == "fish":
            concept = "tuna"
        else:
            concept = concept.split(" ")[-1]  # Usually should handle common names
        glb_path = [
            path
            for path in list_paths
            if concept.lower() in os.path.basename(path).lower()
            and path.endswith(".glb")
            and os.path.basename(path).startswith("paint_mesh")
        ][0]
        glb_path = os.path.join(OUTPUT_DIR, glb_path)
        if not os.path.exists(glb_path):
            return JSONResponse(
                {"error": f"GLB file for '{concept}' not cached yet please generate."},
                status_code=404,
            )

    try:
        extract_textures_and_copy_model(glb_path, CACHE_OUTPUT_DIR)
    except Exception as e:
        return JSONResponse(
            {"error": f"Failed to extract textures: {str(e)}"}, status_code=500
        )

    try:
        zip_folder(CACHE_OUTPUT_DIR, ZIP_PATH)
    except Exception as e:
        return JSONResponse(
            {"error": f"Failed to zip folder: {str(e)}"}, status_code=500
        )

    return FileResponse(
        path=ZIP_PATH, media_type="application/zip", filename=os.path.basename(ZIP_PATH)
    )


if __name__ == "__main__":
    public_url = ngrok.connect(PORT)
    print(f"Public URL: {public_url.public_url}/process-3d-file")

    update_endpoint(public_url.public_url)

    config = uvicorn.Config(app, host="0.0.0.0", port=PORT)
    server = uvicorn.Server(config)

    await server.serve()

Public URL: https://9db3-35-185-249-17.ngrok-free.app/process-3d-file


/tmp/ipykernel_58/4095167237.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_checked": datetime.utcnow().isoformat() + "Z"


Gist updated successfully


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2405:201:4007:b167:f758:2899:4713:c258:0 - "GET / HTTP/1.1" 200 OK
